# Transparency Portal

Brazilian municipal, state and federal governments must, by law, provide data on public spending, budget, and use of federal funds. Of one the tools used for that is "Portal da Transparência" (Transparency Portal), a site that provides such data for public access.

While every municipality, state and federal governments have their own individual Portal da Transparencia, the system behind it varies. Publicenter is a brazilian company that develops a "Portal da Transparencia" system that is used my many brazilian municipalities, including my hometown.

Due to this connection, this project aims to extract information from Publicent's Portal da Transparencia and ease its visualization.

## The study case

The chosen municipality is Lagoa Formosa, a small town located in the interior of the state of Minas Gerais and my hometown.

Lagoa Formosa's Portal da Transparencia can be accessed by this link: [Lagoa Formosa's Portal da Transparência](https://transparencia.lagoaformosa.mg.gov.br/#/transparencia)

This site offers data on many different aspects of public administration, such as, for instance, positions and wages, public contracts, statistical reports, etc., however this project will focus only on public expense data, which is divided budgetary and off-budgetary expenditure.

## The first step: Undestanding HTTP requests

We will begin this project by undestanding how the front-end interact with the system's back-end.
For this we will be using our browser dev tools to inspect HTTP requests and their responses. Lets start with the budgetary expenses:

<p align="center">
  <a href="documentation\images\00-first-header.png">
    <img src="documentation\images\01-first-response.png" width="45%">
  </a>
  <a href="documentation\images\01-first-response.png">
    <img src="documentation\images\01-first-response.png" width="45%">
  </a>
</p>

From this we can get the request url:
`https://transparencia.lagoaformosa.mg.gov.br/publico/despesaDetalhada?elementosPorPagina=20&pagina=1&termoBase64=&indAgrupamento=FOR&datInicio=2026-07-01&datFim=2026-07-31&desNatureza=&codAdministracao=1&filtrarApenasCovid=false&numEmpenho=`

By analysing it we can notice some interesting query parameters, more specifically:
- `elementosPorPagina=` which decribes how many elements will be returned
- `pagina=`, which describes the page requested
- `datInicio=`, that is the starting date
- `datFim=`, that is the ending date

By analysing the response we found tree important keys:
- `"content"`, which is a list of entries
- `"totalElements"`, that is a number of the total number of data entries
- `"totalPages"`, that lists the number of pages
- `"size"`, that is the size of each page

With that we can begin playing with HTTP requests:


In [1]:
# we opted to use the httpx library instead of the requests library
import httpx

elementsPerPage = 20
page = 1
initialDate = '2010-01-01'
finalDate = '2026-07-29'

url = f'https://transparencia.lagoaformosa.mg.gov.br/publico/despesaDetalhada?elementosPorPagina={elementsPerPage}&pagina={page}&termoBase64=&indAgrupamento=FOR&datInicio={initialDate}&datFim={finalDate}&desNatureza=&codAdministracao=1&filtrarApenasCovid=false&numEmpenho='

response = httpx.get(url, verify=False)
r = response.json()
totalElements = r['totalElements']
totalPages = r['totalPages']
size = r['size']
print(f'totalElements: {totalElements}, totalPages: {totalPages}, size: {size}')

totalElements: 40559, totalPages: 2028, size: 20


## The second step: Extracting data from Portal da Transparência

From this we can see that we have a total of 40487 entries distributed among 2025 pages.

We can play with the query parameters to try to reduce the amount of pages, and consequently reduce the number of requests we'll have to make to obtain all data.

Of course, we need to be careful to not trigger a timeout or any other block.

In [2]:
elementsPerPage = 10000

url = f'https://transparencia.lagoaformosa.mg.gov.br/publico/despesaDetalhada?elementosPorPagina={elementsPerPage}&pagina={page}&termoBase64=&indAgrupamento=FOR&datInicio={initialDate}&datFim={finalDate}&desNatureza=&codAdministracao=1&filtrarApenasCovid=false&numEmpenho='

response = httpx.get(url, verify=False)
r = response.json()
totalElements = r['totalElements']
totalPages = r['totalPages']
size = r['size']
print(f'totalElements: {totalElements}, totalPages: {totalPages}, size: {size}')

totalElements: 40559, totalPages: 5, size: 10000


We can see that by setting the number of elements per page as 10000 we reduced the number of pages from 2025 to only 5.

There's not a magical number, so we always need to verify how many elements per page the systems support.

Anyway, we can now capture the data:

In [3]:
elementsPerPage = 10000
initialDate = '2010-01-01'
finalDate = '2026-07-29'

content = []
page, totalPages, totalElements = 1, 1, 1

while (len(content) < totalElements) and (page <= totalPages):

    url = f'https://transparencia.lagoaformosa.mg.gov.br/publico/despesaDetalhada?elementosPorPagina={elementsPerPage}&pagina={page}&termoBase64=&indAgrupamento=FOR&datInicio={initialDate}&datFim={finalDate}&desNatureza=&codAdministracao=1&filtrarApenasCovid=false&numEmpenho='
    response = httpx.get(url, verify=False)
    r = response.json()
    content.extend(r['content'])
    
    if page == 1:
        totalElements = r['totalElements']
        totalPages = r['totalPages']
    
    page += 1

    print(f'totalElements: {totalElements}, elements obtained: {len(content)}')




totalElements: 40559, elements obtained: 10000
totalElements: 40559, elements obtained: 20000
totalElements: 40559, elements obtained: 30000
totalElements: 40559, elements obtained: 40000
totalElements: 40559, elements obtained: 40559


## Third step: converting data for a tabular format

Now we have a list made of 40487 dictionaries, however it is still hard to work with data in this structure. For this reason we will convert it for a tabular format.

We could assume that every dictionary has the same keys to ease the extraction, but we'll check every entry to garantee no data is left behind.

To garantee we'll not need to request the data again, we'll also save it as an csv file, so we can open it latter if needed.

In [4]:
dictKeys = set()

for i in range(len(content)):
    dictKeys.update(content[i].keys())

extractedBudgetaryData = [list(dictKeys)]
for i in range(len(content)):
    row = []
    for j in dictKeys:
        row.append(content[i][j])
    extractedBudgetaryData.append(row)

import csv
with open ('extractedBudgetaryData.csv', 'w', newline="", encoding='utf-8') as file:
    writer = csv.writer(file, delimiter=';')
    writer.writerows(extractedBudgetaryData)

## Doing the same for the off-budgetary expenditure.

We can notice that the request url is pretty similar: `https://transparencia.lagoaformosa.mg.gov.br/publico/despesaExtra?elementosPorPagina=20&pagina=1&termoBase64=&datInicio=2026-07-06&datFim=2026-07-31`

So we only need to work with the some parameters as before.

In [5]:
elementsPerPage = 1000
initialDate = '2010-01-01'
finalDate = '2026-07-29'

content = []
page, totalPages, totalElements = 1, 1, 1

while (len(content) < totalElements) and (page <= totalPages):
    
    url = f'https://transparencia.lagoaformosa.mg.gov.br/publico/despesaExtra?elementosPorPagina={elementsPerPage}&pagina={page}&termoBase64=&datInicio={initialDate}&datFim={finalDate}'
    response = httpx.get(url, verify=False)
    r = response.json()
    content.extend(r['content'])
    
    if page == 1:
        totalElements = r['totalElements']
        totalPages = r['totalPages']    
    page += 1
    print(f'totalElements: {totalElements}, elements obtained: {len(content)}')

dictKeys = set()

for i in range(len(content)):
    dictKeys.update(content[i].keys())

extractedOffBudgetaryData = [list(dictKeys)]
for i in range(len(content)):
    row = []
    for j in dictKeys:
        row.append(content[i][j])
    extractedOffBudgetaryData.append(row)

import csv
with open ('extractedOffBudgetaryData.csv', 'w', newline="", encoding='utf-8') as file:
    writer = csv.writer(file, delimiter=';')
    writer.writerows(extractedOffBudgetaryData)


totalElements: 2831, elements obtained: 1000
totalElements: 2831, elements obtained: 2000
totalElements: 2831, elements obtained: 2831


## Fourth step: Processing data
Now that we have extracted the data, we can work with it, and for this we have many options. 

In this project we'll use two: pandas and metabase:
- Pandas is a powerfull library focused on the manipulation, cleaning, and analysis of structured tabular data.
- Jupyter SQL magic allows us to execute SQL queries directly within Jupyter Notebook cells and display results interactively.
- Metabase is a user-friendly Business Intelligence (BI) tool that allows you to explore data, create charts, and build interactive dashboards, with or without SQL code.

In Pandas we'll be manipulating the data directly in this notebook, as for metabase and SQL magic, we'll convert the data into a sqlite database file and import it directly on metabase and process it using SQL Magic.

In [6]:
import pandas as pd
import sqlite3
import hashlib
import re

budgetDf = pd.DataFrame(extractedBudgetaryData[1:], columns=extractedBudgetaryData[0])
offBudgetDf = pd.DataFrame(extractedOffBudgetaryData[1:], columns=extractedOffBudgetaryData[0])


### 4.1 - Understanding and cleaning the data
Now we need to undestand and clean our data. This step is an essential for every data analysis.

Will also make a small data treatment using pandas prior to exporting it to a sqlite file.

Observation: *I noticed that the data contains sensitive data of natural persons, more specifically the CPF, equivalent to the USA's social security number. For this reason we'll replace such data with a sha-256 hash, this way we'll still be able to agregate data without compromising sensitive personal data.*

In [7]:
# Removing Personal data
cpfRegex = re.compile(r'^\d{3}\.\d{3}\.\d{3}-\d{2}$')

def applyHashCpf(df, column):
    mask = df[column].str.match(cpfRegex, na=False)
    df.loc[mask, column] = (
        df.loc[mask, column]
          .apply(lambda x: hashlib.sha256(x.encode("utf-8")).hexdigest())
    )

applyHashCpf(budgetDf, "desDocumentoFornecedor")
applyHashCpf(offBudgetDf, "numDocumentoCredor")

# Removing empty columns and rows
def dropNaDf(df):
    df.dropna(axis=1, how="all", inplace = True)
    df.dropna(axis=0, how="all", inplace = True)

dropNaDf(budgetDf)
dropNaDf(offBudgetDf)

# Removing columns where all entries have the same value (as they are irrelevant)
budgetDf = budgetDf.loc[:, budgetDf.nunique() > 1]
offBudgetDf = offBudgetDf.loc[:, offBudgetDf.nunique() > 1]


Checking the data structure

In [8]:
# budgetDf.head(10)
budgetDf.info()

<class 'pandas.DataFrame'>
RangeIndex: 40559 entries, 0 to 40558
Data columns (total 25 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   mostrarLicitacao                 40559 non-null  bool   
 1   vlrDescontoInss                  40559 non-null  float64
 2   desLicitacaoEmpenhoFormatado     40559 non-null  str    
 3   desProcessoEmpenho               40559 non-null  str    
 4   vlrMovimento                     40559 non-null  str    
 5   totalDesconto                    40559 non-null  float64
 6   numEmpenho                       40559 non-null  str    
 7   vlrLiquidoPago                   35356 non-null  float64
 8   desLicitacaoEmpenho              40559 non-null  str    
 9   fornecedorFormatado              40559 non-null  str    
 10  desDocumentoFornecedorFormatado  40559 non-null  str    
 11  vlrLiquidadoFormatado            40559 non-null  str    
 12  vlrDescontoOutros            

In [9]:
# budgetDf.head(10)
offBudgetDf.info()

<class 'pandas.DataFrame'>
RangeIndex: 2831 entries, 0 to 2830
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   desClassificacao            2831 non-null   str    
 1   nomCredor                   2831 non-null   str    
 2   numDocumentoCredor          2831 non-null   str    
 3   vlrDespesa                  2831 non-null   float64
 4   codDespesaExtra             2831 non-null   int64  
 5   desConta                    2831 non-null   str    
 6   fornecedorFormatado         2831 non-null   str    
 7   datMovimento                2831 non-null   str    
 8   numDocumentCredorFormatado  2831 non-null   str    
dtypes: float64(1), int64(1), str(7)
memory usage: 664.7 KB


After analysing the data visually we found the following column have correspondent data: 
- Budgetary Data
    - desDocumentoFornecedorFormatado ~ fornecedorFormatado
    - numEmpenho & numAnoEmpenho = empenhoFormatado
    - desLicitacaoEmpenhoFormatado ~ desLicitacaoEmpenho
    - vlrEmpenhado = vlrEmpenhadoFormatado & vlrMovimento
    - vlrLiquidado = vlrLiquidadoFormatado
    - vlrPago ~ vlrPagoFormatado
- Off Budgetary Data
    - nomCredor & numDocumentCredorFormatado = fornecedorFormatado

Thus they will be removed to avoid redundant data (the ones in the left will be kept)

In [10]:
budgetDf.drop(columns=['fornecedorFormatado', 'empenhoFormatado', 'desLicitacaoEmpenho', 'vlrEmpenhadoFormatado', 'vlrMovimento', 'vlrLiquidadoFormatado', 'vlrPagoFormatado'], inplace = True)
offBudgetDf.drop(columns=['fornecedorFormatado'], inplace = True)

We also found that the column "desModalidade" of budgetDf contains similar data, but i needs to be processed/adjusted:

In [11]:
budgetDf['desModalidade'].unique()

<ArrowStringArray>
[                              'Dispensada',
                             'Concorrencia',
 'INEXIGIBILIDADE CREDENCIA/CHAM. PÃšBLICO',
                      'PREGÃƒO ELETRÃ”NICO',
                        'Tomada de PreÃ§os',
                      'PregÃ£o EletrÃ´nico',
                                  'PregÃ£o',
                          'INEXIGIBILIDADE',
                            'ConcorrÃªncia',
                                 'DISPENSA',
       'INEXIGIBILIDADE POR CREDENCIAMENTO',
                          'Inexigibilidade',
                                 'Dispensa',
                                  'Convite',
                            'Carta Convite']
Length: 15, dtype: str

In [12]:
equivalenceDict = {
    'DISPENSA': 'Dispensa',
    'Dispensada': 'Dispensa',
    'Tomada de PreÃ§os' : 'Tomada de Precos',
    'PREGÃƒO ELETRÃ”NICO' : 'Pregao Eletronico',
    'PregÃ£o EletrÃ´nico' : 'Pregao Eletronico',
    'PregÃ£o' : 'Pregao',
    'INEXIGIBILIDADE' : 'Inexigibilidade',
    'INEXIGIBILIDADE POR CREDENCIAMENTO' : 'Inexigibilidade',
    'INEXIGIBILIDADE CREDENCIA/CHAM. PÃšBLICO' : 'Inexigibilidade',
    'ConcorrÃªncia' : 'Concorrencia', 
    'Carta Convite' : 'Convite'
}

budgetDf["desModalidade"] = budgetDf["desModalidade"].replace(equivalenceDict)
budgetDf['desModalidade'].unique()


<ArrowStringArray>
[         'Dispensa',      'Concorrencia',   'Inexigibilidade',
 'Pregao Eletronico',  'Tomada de Precos',            'Pregao',
           'Convite']
Length: 7, dtype: str

Now the data is good enough to be exported / saved:

In [13]:
# Exporting data

# We'll be rewriting the previously generated CSVs in order to not compromise personal data
budgetDf.to_csv("extractedBudgetaryData.csv", sep=";", index=False)
offBudgetDf.to_csv("extractedOffBudgetaryData.csv", sep=";", index=False)

# We'll be using pandas itself to connect to the sqlite and crete the tables:
dbconn = sqlite3.connect('portalDaTransparencia.sqlite')

budgetDf.to_sql(name='budgetData', con=dbconn, if_exists='replace', index=False)
offBudgetDf.to_sql(name='offBudgetData', con=dbconn, if_exists='replace', index=False)

dbconn.close()

print(f'length budgetDf: {len(budgetDf)}, length budgetDf: {len(offBudgetDf)}')

length budgetDf: 40559, length budgetDf: 2831


If we need to read the data again from the files we can just:

In [14]:
import pandas as pd
import sqlite3
budgetDf = pd.read_csv("extractedBudgetaryData.csv", sep=";")
offBudgetDf = pd.read_csv("extractedOffBudgetaryData.csv", sep=";")
# dbconn = sqlite3.connect('portalDaTransparencia.sqlite')


### 4.2 - Extracting relevant information
And now we can begin processing and extracting relevant information from our data.

We'll generate a ranking of the top 10 recipient entities based on the following metrics:

* Total amount of funds received.
* Number of incoming transactions.
* Average amount received per payment (total amount received ÷ number of payments received).

The rankings will be generated annually and for the full duration of the dataset.

All of these will be generated using Pandas, SQL magic, and on metabase

Lets start by analysing the columns and their data type:

In [15]:
budgetDf.info()

<class 'pandas.DataFrame'>
RangeIndex: 40559 entries, 0 to 40558
Data columns (total 18 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   mostrarLicitacao                 40559 non-null  bool   
 1   vlrDescontoInss                  40559 non-null  float64
 2   desLicitacaoEmpenhoFormatado     40559 non-null  str    
 3   desProcessoEmpenho               40559 non-null  str    
 4   totalDesconto                    40559 non-null  float64
 5   numEmpenho                       40559 non-null  int64  
 6   vlrLiquidoPago                   35356 non-null  float64
 7   desDocumentoFornecedorFormatado  40559 non-null  str    
 8   vlrDescontoOutros                40559 non-null  float64
 9   vlrPago                          40559 non-null  float64
 10  desModalidade                    40559 non-null  str    
 11  desDocumentoFornecedor           40559 non-null  str    
 12  desAgrupamento               

#### 4.2.2 - Extracting relevant information with pandas


In [16]:
def aggregateBudgetaryData(df):
    budgetDfAggData = df.groupby("desDocumentoFornecedor", as_index=False).agg(
        desAgrupamento=("desAgrupamento", "first"),
        numberOfTransactions = ("desDocumentoFornecedor", "count"),
        committedValue = ("vlrEmpenhado", "sum")
    )
    budgetDfAggData['averageReceived'] = budgetDfAggData['committedValue'] / budgetDfAggData['numberOfTransactions']

    return budgetDfAggData.copy()

# We'll create a dictionary to store all the dataframes with the aggregated information
budgetDict = {}

budgetDict['agg-AllTime-Allmodality'] = aggregateBudgetaryData(budgetDf)

for year in budgetDf['numAnoEmpenho'].unique():
    filter = budgetDf[budgetDf['numAnoEmpenho'] == year]
    budgetDict[f'agg-{year}-Allmodality'] = aggregateBudgetaryData(filter)

for modality in budgetDf['desModalidade'].unique():
    filter = budgetDf[budgetDf['desModalidade'] == modality]
    budgetDict[f'agg-AllTime-{modality}'] = aggregateBudgetaryData(filter)

for year in budgetDf['numAnoEmpenho'].unique():
    for modality in budgetDf['desModalidade'].unique():
        filter = budgetDf[(budgetDf['numAnoEmpenho'] == year) & (budgetDf['desModalidade'] == modality)]
        budgetDict[f'agg-{year}-{modality}'] = aggregateBudgetaryData(filter)



Now we can display all dataframes generated.

With them we can check who are the entities that they received more resources from the municipality.

In [17]:
from IPython.display import display, Markdown

def brl(x):
    return f"R$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

currencyFormat = {
    "committedValue": brl,
    "averageReceived": brl
}


for key, df in budgetDict.items():
    df.to_csv(f'extractedData/{key}.csv', sep=";") # Will also export the data as csv

    # The code bellow shows all dataframes, how ever we'll show only a few selected ones to avoid visual polution
    # for column in ["numberOfTransactions", "committedValue", "averageReceived"]:
    #     display(Markdown(f"#### {key} - {column}"))
    #     display(df.sort_values(column, ascending=False).head(10).style.format(currencyFormat))

selectedDfs = ['agg-AllTime-Allmodality', 'agg-AllTime-Inexigibilidade', 'agg-AllTime-Dispensa']

for key in selectedDfs:
    for column in ["committedValue", "averageReceived"]:
        display(Markdown(f"#### {key} - {column}"))
        display(budgetDict[key].sort_values(column, ascending=False).head(10).style.format(currencyFormat))


#### agg-AllTime-Allmodality - committedValue

,desDocumentoFornecedor,desAgrupamento,numberOfTransactions,committedValue,averageReceived
69,02.319.394/0001-70,02.319.394/0001-70 - CISALP CONSORCIO PUBLICO INTERMUNICIPAL DE SAUDE A MICRO REG,7800,"R$ 172.229.378,03","R$ 22.080,69"
484,18.602.078/0001-41,18.602.078/0001-41 - MUNICIPIO DE LAGOA FORMOSA,2897,"R$ 69.441.769,18","R$ 23.970,23"
627,23.096.837/0001-81,23.096.837/0001-81 - SISTEMA DE BENEFICIENCIA DOS SERV PUBL MUN LAGOA FORMOS,1004,"R$ 45.463.254,67","R$ 45.282,13"
268,09.415.124/0001-02,09.415.124/0001-02 - SOLID CONTRUTORA E SERVICOS LTDA,733,"R$ 26.482.197,94","R$ 36.128,51"
24,00.604.122/0001-97,00.604.122/0001-97 - TRIVALE INSTITUICAO DE PAGAMENTO LTDA,494,"R$ 15.469.901,50","R$ 31.315,59"
795,29.979.036/0001-40,29.979.036/0001-40 - INSTITUTO NACIONAL DO SEGURO SOCIAL,3095,"R$ 14.739.701,25","R$ 4.762,42"
187,06.981.180/0001-16,06.981.180/0001-16 - CEMIG DISTRIBUICAO S A,219,"R$ 8.539.291,35","R$ 38.992,20"
11,00.360.305/3540-03,00.360.305/3540-03 - CAIXA ECONOMICA FEDERAL,38,"R$ 8.496.191,65","R$ 223.583,99"
1057,41.804.972/0001-16,41.804.972/0001-16 - ESTRELA INFRAESTRUTURA LTDA,51,"R$ 7.729.309,97","R$ 151.555,10"
3,00.207.500/0001-07,00.207.500/0001-07 - MULTIMEDIC COMERCIAL LTDA,362,"R$ 7.442.076,62","R$ 20.558,22"


#### agg-AllTime-Allmodality - averageReceived

,desDocumentoFornecedor,desAgrupamento,numberOfTransactions,committedValue,averageReceived
277,0ab04d92f1a7041d7611f58c12e8b24e4afee201ff0d404b5ca268be89a8ac5f,***.761.606-** - MARIA MOREIRA DE JESUS,1,"R$ 1.100.000,00","R$ 1.100.000,00"
552,20.495.149/0001-04,20.495.149/0001-04 - PRODOESTE VEICULOS E SERVICOS LTDA.,1,"R$ 1.032.000,00","R$ 1.032.000,00"
739,27.469.763/0001-32,27.469.763/0001-32 - PLANARES CONSTRUTORA LTDA.,2,"R$ 2.000.000,00","R$ 1.000.000,00"
850,31.534.481/0001-49,31.534.481/0001-49 - DMX CONSTRUTORA LTDA,3,"R$ 2.440.876,80","R$ 813.625,60"
1009,39.592.941/0001-05,39.592.941/0001-05 - BUSMASTER LOCADORA E DISTRIBUIDORA DE VEICULOS EIRELI,4,"R$ 2.729.500,00","R$ 682.375,00"
1273,53.007.444/0001-15,53.007.444/0001-15 - CONSORCIO PFM - REURB,1,"R$ 565.190,27","R$ 565.190,27"
994,38.392.035/0001-96,38.392.035/0001-96 - MEDIACAO DO MORAR GESTAO DE CONFLITOS LTDA,4,"R$ 2.237.620,00","R$ 559.405,00"
77,02.659.246/0001-03,02.659.246/0001-03 - VMI TECNOLOGIAS LTDA.,1,"R$ 525.746,66","R$ 525.746,66"
538,20.015.459/0001-76,20.015.459/0001-76 - SINDICATO DOS PRODUTORES RURAIS DE LAGOA FORMOSA,2,"R$ 980.000,00","R$ 490.000,00"
852,31.564.854/0001-24,31.564.854/0001-24 - BARROS BITTENCOURT EMPREENDIMENTOS LTDA,7,"R$ 3.316.052,56","R$ 473.721,79"


#### agg-AllTime-Inexigibilidade - committedValue

...

,desDocumentoFornecedor,desAgrupamento,numberOfTransactions,committedValue,averageReceived
26,08.829.480/0001-00,08.829.480/0001-00 - W M SHOWS LTDA,1,"R$ 198.000,00","R$ 198.000,00"
46,21.766.859/0001-86,21.766.859/0001-86 - OLIVEIRA E LIMA ASSISTENCIA MEDICA LTDA,9,"R$ 1.573.576,34","R$ 174.841,82"
84,342b82c865c9e87925a137c2ad39756e88e565ff73d00bf07e7d3b2385bf218a,***.174.956-** - SERGIO ANTONIO TEIXEIRA,2,"R$ 344.027,82","R$ 172.013,91"
23,08.111.952/0001-94,08.111.952/0001-94 - M & P FERREIRA PRODUCOES LTDA,1,"R$ 140.000,00","R$ 140.000,00"
89,37.232.761/0001-89,37.232.761/0001-89 - AMISTAD LOCACOES E EVENTOS LTDA,1,"R$ 140.000,00","R$ 140.000,00"
86,36.574.431/0001-09,36.574.431/0001-09 - RB PRODUCOES ARTISTICAS LTDA,1,"R$ 135.000,00","R$ 135.000,00"
160,d7ba2a81f60f68f44ef0d5b5dfa424ece815e94bf94b237e67573b7f8f11d14f,***.883.706-** - JOSE BATISTA DE OLIVEIRA,2,"R$ 268.200,00","R$ 134.100,00"
121,59.826.721/0001-06,59.826.721/0001-06 - 59.826.721 RAGNES DONIZETTE DE OLIVEIRA,1,"R$ 132.006,00","R$ 132.006,00"
47,22.300.057/0001-49,22.300.057/0001-49 - RICARDO GERALDO PONTELO,8,"R$ 1.055.368,00","R$ 131.921,00"
22,08.111.069/0001-02,08.111.069/0001-02 - RJ GESTAO EM NEGOCIOS LTDA,4,"R$ 499.000,00","R$ 124.750,00"


#### agg-AllTime-Dispensa - committedValue

,desDocumentoFornecedor,desAgrupamento,numberOfTransactions,committedValue,averageReceived
49,02.319.394/0001-70,02.319.394/0001-70 - CISALP CONSORCIO PUBLICO INTERMUNICIPAL DE SAUDE A MICRO REG,7800,"R$ 172.229.378,03","R$ 22.080,69"
287,18.602.078/0001-41,18.602.078/0001-41 - MUNICIPIO DE LAGOA FORMOSA,2897,"R$ 69.441.769,18","R$ 23.970,23"
376,23.096.837/0001-81,23.096.837/0001-81 - SISTEMA DE BENEFICIENCIA DOS SERV PUBL MUN LAGOA FORMOS,1004,"R$ 45.463.254,67","R$ 45.282,13"
456,29.979.036/0001-40,29.979.036/0001-40 - INSTITUTO NACIONAL DO SEGURO SOCIAL,3095,"R$ 14.739.701,25","R$ 4.762,42"
10,00.360.305/3540-03,00.360.305/3540-03 - CAIXA ECONOMICA FEDERAL,38,"R$ 8.496.191,65","R$ 223.583,99"
12,00.394.460/0001-41,00.394.460/0001-41 - MINISTERIO DA FAZENDA,56,"R$ 6.152.474,25","R$ 109.865,61"
117,06.981.180/0001-16,06.981.180/0001-16 - CEMIG DISTRIBUICAO S A,70,"R$ 2.600.837,89","R$ 37.154,83"
378,23.114.937/0001-93,23.114.937/0001-93 - ASSOC DE PAIS E AMIGOS DOS EXCEPCIONAIS,22,"R$ 2.057.756,13","R$ 93.534,37"
654,49.013.630/0001-90,49.013.630/0001-90 - CONSTRUTORA E SERVICOS LJ LTDA,7,"R$ 1.874.850,00","R$ 267.835,71"
527,35.992.000/0001-08,35.992.000/0001-08 - CONSELHO COMUNITARIO DE SEGURANCA PUBLICA DE LAGOA FORMOSA,6,"R$ 1.833.000,08","R$ 305.500,01"


#### agg-AllTime-Dispensa - averageReceived

,desDocumentoFornecedor,desAgrupamento,numberOfTransactions,committedValue,averageReceived
165,0ab04d92f1a7041d7611f58c12e8b24e4afee201ff0d404b5ca268be89a8ac5f,***.761.606-** - MARIA MOREIRA DE JESUS,1,"R$ 1.100.000,00","R$ 1.100.000,00"
322,20.015.459/0001-76,20.015.459/0001-76 - SINDICATO DOS PRODUTORES RURAIS DE LAGOA FORMOSA,2,"R$ 980.000,00","R$ 490.000,00"
969,b4fde3ad5194701331c201232fdd94af518eae1ff92a77914614a49b64c52381,***.607.126-** - VAGNER GONCALVES PEREIRA,1,"R$ 350.000,00","R$ 350.000,00"
527,35.992.000/0001-08,35.992.000/0001-08 - CONSELHO COMUNITARIO DE SEGURANCA PUBLICA DE LAGOA FORMOSA,6,"R$ 1.833.000,08","R$ 305.500,01"
241,15.984.883/0001-99,15.984.883/0001-99 - ELETRICA RADIANTE MAT ELETRICOS LTDA,2,"R$ 588.171,45","R$ 294.085,73"
654,49.013.630/0001-90,49.013.630/0001-90 - CONSTRUTORA E SERVICOS LJ LTDA,7,"R$ 1.874.850,00","R$ 267.835,71"
10,00.360.305/3540-03,00.360.305/3540-03 - CAIXA ECONOMICA FEDERAL,38,"R$ 8.496.191,65","R$ 223.583,99"
477,30.904.666/0001-35,30.904.666/0001-35 - IRMÃOS LEAL LTDA,6,"R$ 1.289.296,00","R$ 214.882,67"
166,0ae134d452ea646adcd7d4ed676157e1c38f4e444852ff0e68240f892cb3a76a,***.454.276-** - CARLOS PRATES MORAIS,1,"R$ 182.660,56","R$ 182.660,56"
188,11.333.479/0001-02,11.333.479/0001-02 - FUNDO MUNICIPAL DE SAUDE DE LAGOA FORMOSA,11,"R$ 1.800.046,49","R$ 163.640,59"


Another interesting aggregation is on the "desModalidade" data, as ir shows data by different procurement methods:
| Portuguese                                  | English                                                   | Simple Explanation                                                                                                                                                                               |
| ------------------------------------------- | --------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| **Licitação**                               | **Public Procurement (Bidding Process)**                  | The general process used by the government to choose the best supplier in a fair and transparent way.                                                                                            |
| **Pregão**                                  | **Auction / Reverse Auction**                             | A competitive bidding method where suppliers compete mainly by offering the lowest price for standard goods and services.                                                                        |
| **Pregão Eletrônico**                       | **Electronic Reverse Auction**                            | The same as a *Pregão*, but conducted entirely online. Suppliers submit and improve their bids electronically.                                                                                   |
| **Concorrência**                            | **Competitive Tender**                                    | A formal procurement method used for larger or more complex contracts, such as construction projects or high-value purchases.                                                                    |
| **Convite** *(repealed by Law 14.133/2021)* | **Invitation Bidding**                                    | An older method where the government invited at least three suppliers to submit proposals. No longer used under the current procurement law.                                                     |
| **Tomada de Preços** *(repealed)*           | **Restricted Tender**                                     | An older procedure limited to previously registered suppliers or those who met registration requirements before the deadline.                                                                    |
| **Concurso**                                | **Contest**                                               | Used when selecting the best technical, artistic, scientific, or architectural work, often with a prize for the winner.                                                                          |
| **Leilão**                                  | **Public Auction**                                        | Used when the government wants to sell assets, such as vehicles, equipment, or real estate, to the highest bidder.                                                                               |
| **Dispensa de Licitação**                   | **Waiver of Competitive Bidding**                         | The law allows the government to buy directly without a bidding process in specific situations, such as low-value purchases or emergencies. Competition is possible but not legally required.    |
| **Inexigibilidade de Licitação**            | **Non-Competitive Procurement (Sole Source Procurement)** | Competition is impossible because there is only one suitable supplier or the service is unique, such as hiring a renowned artist or purchasing proprietary software from its exclusive provider. |


Data bellow shows that over the years the "Dispensa" method was by far the one that reveived the most resources. As it do not have a bidding process, we can say that it may be more susceptible to corruption, although that in itself is not an indication of it.

In [18]:
from IPython.display import display, Markdown

def aggregateBudgetaryData(df):
    budgetDfAggData = df.groupby("desModalidade", as_index=False).agg(
        numberOfTransactions = ("desDocumentoFornecedor", "count"),
        committedValue = ("vlrEmpenhado", "sum")
    )
    budgetDfAggData['averageReceived'] = budgetDfAggData['committedValue'] / budgetDfAggData['numberOfTransactions']

    return budgetDfAggData.copy()


# We'll create a dictionary to store all the dataframes with the aggregated information
budgetDict2 = {}

budgetDict2['agg-Modality-AllTime'] = aggregateBudgetaryData(budgetDf)

for year in budgetDf['numAnoEmpenho'].unique():
    filter = budgetDf[(budgetDf['numAnoEmpenho'] == year)]
    budgetDict2[f'agg-Modality-{year}'] = aggregateBudgetaryData(filter)

def brl(x):
    return f"R$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

currencyFormat = {
    "committedValue": brl,
    "averageReceived": brl
}

for key, df in budgetDict2.items():
    df.to_csv(f'extractedData/{key}.csv', sep=";", index=False) # Will also export the data as csv

    # The code bellow shows all dataframes, how ever we'll show only a few selected ones to avoid visual polution
    # for column in ["numberOfTransactions", "committedValue", "averageReceived"]:
    #     display(Markdown(f"#### {key} - {column}"))
    #     display(df.sort_values(column, ascending=False).style.format(currencyFormat))

selectedDfs = ['agg-Modality-AllTime']

for key in selectedDfs:
    for column in ["committedValue", "averageReceived"]:
        display(Markdown(f"#### {key} - {column}"))
        display(budgetDict2[key].sort_values(column, ascending=False).head(10).style.format(currencyFormat))


#### agg-Modality-AllTime - committedValue

,desModalidade,numberOfTransactions,committedValue,averageReceived
2,Dispensa,24954,"R$ 348.161.824,19","R$ 13.952,14"
5,Pregao Eletronico,8231,"R$ 77.574.138,10","R$ 9.424,63"
4,Pregao,4361,"R$ 52.247.511,87","R$ 11.980,63"
0,Concorrencia,421,"R$ 41.094.899,20","R$ 97.612,59"
6,Tomada de Precos,135,"R$ 23.749.833,10","R$ 175.924,69"
3,Inexigibilidade,1406,"R$ 22.918.015,59","R$ 16.300,15"
1,Convite,1051,"R$ 4.844.054,48","R$ 4.609,00"


#### agg-Modality-AllTime - averageReceived

,desModalidade,numberOfTransactions,committedValue,averageReceived
6,Tomada de Precos,135,"R$ 23.749.833,10","R$ 175.924,69"
0,Concorrencia,421,"R$ 41.094.899,20","R$ 97.612,59"
3,Inexigibilidade,1406,"R$ 22.918.015,59","R$ 16.300,15"
2,Dispensa,24954,"R$ 348.161.824,19","R$ 13.952,14"
4,Pregao,4361,"R$ 52.247.511,87","R$ 11.980,63"
5,Pregao Eletronico,8231,"R$ 77.574.138,10","R$ 9.424,63"
1,Convite,1051,"R$ 4.844.054,48","R$ 4.609,00"


We can check, for instance, how the commited value varied over the total expenditure over the years.

By analysing the data we can notice that for every single year the "dispensa" was by far the most used procurement method.

In [19]:
budgetDict3 = budgetDict2.copy()

colNames = []

for key, df in budgetDict3.items():    
    newColName = f'committedValue-{key.split('-')[-1]}%'
    colNames.append(newColName)
    df[newColName] = (df['committedValue'] / df['committedValue'].sum()) * 100
    budgetDict3[key] = df[['desModalidade', newColName]]

from functools import reduce

colNames.sort()
colNames.insert(0, 'desModalidade')

modalityOverTimeDf = reduce(lambda left, right: pd.merge(left, right, on='desModalidade', how='left'), list(budgetDict3.values()))

modalityOverTimeDf = modalityOverTimeDf[colNames].sort_values("committedValue-AllTime%", ascending=False)

modalityOverTimeDf.to_csv(f'extractedData/{key}.csv', sep=";", index=False) # Will also export the data as csv

modalityOverTimeDf

,desModalidade,committedValue-2021%,committedValue-2022%,committedValue-2023%,committedValue-2024%,committedValue-2025%,committedValue-2026%,committedValue-AllTime%
2,Dispensa,67.698518,54.621885,54.420835,62.182800,66.610666,62.236509,61.017833
5,Pregao Eletronico,9.085117,21.943407,18.211686,14.009394,9.512202,6.837356,13.595419
4,Pregao,9.435696,17.132727,13.660319,5.063681,5.346612,5.109709,9.156748
0,Concorrencia,NaN,0.215230,2.014908,10.408714,11.185983,17.198210,7.202173
6,Tomada de Precos,1.644196,3.935351,8.191328,5.356099,2.524118,1.831262,4.162327
3,Inexigibilidade,4.542533,2.048134,3.500924,2.979312,4.820419,6.786954,4.016545
1,Convite,7.593940,0.103266,NaN,NaN,NaN,NaN,0.848955


we can also plot this data:

In [20]:
import plotly.express as px

dfLong = modalityOverTimeDf.melt(
    id_vars="desModalidade",
    var_name="Year",
    value_name="Percentage"
)

dfLong["Year"] = dfLong["Year"].str.replace("committedValue-", "").str.replace("%", "")
dfLong = dfLong.dropna()

fig = px.bar(
    dfLong,
    x="Year",
    y="Percentage",
    color="desModalidade",
    title="Procurement methods preference",
    barmode="stack"
)

fig.update_layout(
    template="plotly_white",
    yaxis=dict(
        title="Percentage (%)",
        range=[0,100],
        ticksuffix="%"
    ),
    xaxis=dict(
        title="Year"
    ),
    hovermode="x unified"
)
fig.write_image("documentation/images/02-newplot.png")
fig.show()

Bellow we have the graph generated

![](documentation/images/02-newplot.png)

#### The same process can be performed on the off budget data

However, as the data structure is different a couple changes must me made.

As we already made many step by step comments in markdown, instead I'll use comments directly on code

In [21]:
offBudgetDf.info()

<class 'pandas.DataFrame'>
RangeIndex: 2831 entries, 0 to 2830
Data columns (total 8 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   desClassificacao            2831 non-null   int64  
 1   nomCredor                   2831 non-null   str    
 2   numDocumentoCredor          2831 non-null   str    
 3   vlrDespesa                  2831 non-null   float64
 4   codDespesaExtra             2831 non-null   int64  
 5   desConta                    2831 non-null   str    
 6   datMovimento                2831 non-null   str    
 7   numDocumentCredorFormatado  2831 non-null   str    
dtypes: float64(1), int64(2), str(5)
memory usage: 477.1 KB


In [22]:
offBudgetDf.head(5)

,desClassificacao,nomCredor,numDocumentoCredor,vlrDespesa,codDespesaExtra,desConta,datMovimento,numDocumentCredorFormatado
0,218830104,MUNICIPIO DE LAGOA FORMOSA,18.602.078/0001-41,4.74,2047,IRRF PRESTADORES DE SERVIÇOS,2026-07-27,18.602.078/0001-41
1,218850108,MUNICIPIO DE LAGOA FORMOSA,18.602.078/0001-41,94.83,2778,ISSQN PRESTADORES DE SERVIÇOS,2026-07-27,18.602.078/0001-41
2,218830104,MUNICIPIO DE LAGOA FORMOSA,18.602.078/0001-41,1420.36,1428,IRRF PRESTADORES DE SERVIÇOS,2026-07-24,18.602.078/0001-41
3,218830104,MUNICIPIO DE LAGOA FORMOSA,18.602.078/0001-41,34.91,2070,IRRF PRESTADORES DE SERVIÇOS,2026-07-22,18.602.078/0001-41
4,218850108,MUNICIPIO DE LAGOA FORMOSA,18.602.078/0001-41,8389.98,2785,ISSQN PRESTADORES DE SERVIÇOS,2026-07-22,18.602.078/0001-41


In [23]:
# Converting the datMovimento to datetime, so we can filter it latter
offBudgetDf['datMovimento'] = pd.to_datetime(offBudgetDf["datMovimento"],format="%Y-%m-%d")
offBudgetDf['year'] = offBudgetDf['datMovimento'].dt.year

In [24]:
from IPython.display import display, Markdown

def aggregateOffBudgetaryData(df):
    offBudgetDfAggData = df.groupby("numDocumentoCredor", as_index=False).agg(
        nomCredor=("nomCredor", "first"),
        numberOfTransactions = ("numDocumentoCredor", "count"),
        value = ("vlrDespesa", "sum")
    )
    offBudgetDfAggData['averageValue'] = offBudgetDfAggData['value'] / offBudgetDfAggData['numberOfTransactions']

    return offBudgetDfAggData.copy()

offBudgetDict = {}

offBudgetDict['agg-AllTime'] = aggregateOffBudgetaryData(offBudgetDf)

for year in offBudgetDf['year'].unique():
    filter = offBudgetDf[offBudgetDf['year'] == year]
    offBudgetDict[f'agg-{year}'] = aggregateOffBudgetaryData(filter)


def brl(x):
    return f"R$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

currencyFormat = {
    "committedValue": brl,
    "averageReceived": brl
}

selectedDfs = ['agg-AllTime', 'agg-2025']

for key in selectedDfs:
    for column in ["value", "numberOfTransactions"]:
        display(Markdown(f"#### {key} - {column}"))
        display(offBudgetDict[key].sort_values(column, ascending=False).head(10).style.format(currencyFormat))




#### agg-AllTime - value

,numDocumentoCredor,nomCredor,numberOfTransactions,value,averageValue
12,18.602.078/0001-41,MUNICIPIO DE LAGOA FORMOSA,1577,11953511.390000,7579.905764
21,23.096.837/0001-81,SISTEMA DE BENEFICIENCIA DOS SERV PUBL MUN LAGOA FORMOS,93,10515462.290000,113069.486989
25,29.979.036/0001-40,INSTITUTO NACIONAL DO SEGURO SOCIAL,337,10513784.060000,31198.172285
20,23.090.558/0001-00,SINDICATO DOS SERVIDORES PUBLICOS MUNICIPAIS DE LAGOA FORMOS,164,6007250.690000,36629.577378
0,00.000.000/2337-06,BANCO DO BRASIL SA,79,3276986.460000,41480.841266
2,00.360.305/0142-45,CAIXA ECONOMICA FEDERAL,56,759047.540000,13554.420357
1,00.360.305/0001-04,CAIXA ECONOMICA FEDERAL,62,751512.990000,12121.177258
33,87.510.475/0001-06,COOPERATIVA DE CREDITO INTEGRACAO ROTA DAS TERRAS SICREDI,70,203721.520000,2910.307429
38,e42a8d460e7545e00484ff1d11ab3e2d2654b050d0c64376d30b52b76f115466,ALMIRO BRAGA COELHO,1,97552.000000,97552.000000
34,8731826b22315e211012e3b03757a625b61727dfd433564f559225df0670bd01,MAURA IMACULADA GOMES,43,42999.070000,999.978372


#### agg-AllTime - numberOfTransactions

,numDocumentoCredor,nomCredor,numberOfTransactions,value,averageValue
12,18.602.078/0001-41,MUNICIPIO DE LAGOA FORMOSA,1577,11953511.390000,7579.905764
25,29.979.036/0001-40,INSTITUTO NACIONAL DO SEGURO SOCIAL,337,10513784.060000,31198.172285
20,23.090.558/0001-00,SINDICATO DOS SERVIDORES PUBLICOS MUNICIPAIS DE LAGOA FORMOS,164,6007250.690000,36629.577378
21,23.096.837/0001-81,SISTEMA DE BENEFICIENCIA DOS SERV PUBL MUN LAGOA FORMOS,93,10515462.290000,113069.486989
0,00.000.000/2337-06,BANCO DO BRASIL SA,79,3276986.460000,41480.841266
33,87.510.475/0001-06,COOPERATIVA DE CREDITO INTEGRACAO ROTA DAS TERRAS SICREDI,70,203721.520000,2910.307429
41,f3b981b8993d169aeccde778271ff6137a7ca3c84450b4be2189725959817dd6,EDILENE FERNANDES,68,37834.580000,556.390882
16,1e4b979551b336e9a117f072baeb0f34889ab49fa4b524d1a0877891fcc88152,PAOLA TERENCIO BRAGA,66,20200.080000,306.061818
31,5d5c00310f35c8095ca1ed783cefb7ad59cb6465c704b7b7cce4c41043b80775,MISLENE LUSIA OLIVEIRA,66,30737.700000,465.722727
24,26c97a598bf242128d314f605b6e7c38729a09ef54146b53f45eb4fe299cd41d,AURORA LÍDIA DE SANTANA GOMES,65,17300.400000,266.160000


#### agg-2025 - value

,numDocumentoCredor,nomCredor,numberOfTransactions,value,averageValue
4,18.602.078/0001-41,MUNICIPIO DE LAGOA FORMOSA,350,2710797.920000,7745.136914
9,23.096.837/0001-81,SISTEMA DE BENEFICIENCIA DOS SERV PUBL MUN LAGOA FORMOS,16,2464044.440000,154002.777500
11,29.979.036/0001-40,INSTITUTO NACIONAL DO SEGURO SOCIAL,45,2006553.290000,44590.073111
8,23.090.558/0001-00,SINDICATO DOS SERVIDORES PUBLICOS MUNICIPAIS DE LAGOA FORMOS,28,1264939.710000,45176.418214
0,00.000.000/2337-06,BANCO DO BRASIL SA,15,743039.290000,49535.952667
2,00.360.305/0142-45,CAIXA ECONOMICA FEDERAL,17,295608.810000,17388.753529
15,87.510.475/0001-06,COOPERATIVA DE CREDITO INTEGRACAO ROTA DAS TERRAS SICREDI,15,42598.040000,2839.869333
18,f3b981b8993d169aeccde778271ff6137a7ca3c84450b4be2189725959817dd6,EDILENE FERNANDES,14,9378.260000,669.875714
14,5d5c00310f35c8095ca1ed783cefb7ad59cb6465c704b7b7cce4c41043b80775,MISLENE LUSIA OLIVEIRA,12,6338.500000,528.208333
7,23.089.501/0001-91,CAMARA MUNICIPAL DE LAGOA FORMOSA,1,5914.570000,5914.570000


#### agg-2025 - numberOfTransactions

,numDocumentoCredor,nomCredor,numberOfTransactions,value,averageValue
4,18.602.078/0001-41,MUNICIPIO DE LAGOA FORMOSA,350,2710797.920000,7745.136914
11,29.979.036/0001-40,INSTITUTO NACIONAL DO SEGURO SOCIAL,45,2006553.290000,44590.073111
8,23.090.558/0001-00,SINDICATO DOS SERVIDORES PUBLICOS MUNICIPAIS DE LAGOA FORMOS,28,1264939.710000,45176.418214
2,00.360.305/0142-45,CAIXA ECONOMICA FEDERAL,17,295608.810000,17388.753529
9,23.096.837/0001-81,SISTEMA DE BENEFICIENCIA DOS SERV PUBL MUN LAGOA FORMOS,16,2464044.440000,154002.777500
15,87.510.475/0001-06,COOPERATIVA DE CREDITO INTEGRACAO ROTA DAS TERRAS SICREDI,15,42598.040000,2839.869333
0,00.000.000/2337-06,BANCO DO BRASIL SA,15,743039.290000,49535.952667
18,f3b981b8993d169aeccde778271ff6137a7ca3c84450b4be2189725959817dd6,EDILENE FERNANDES,14,9378.260000,669.875714
14,5d5c00310f35c8095ca1ed783cefb7ad59cb6465c704b7b7cce4c41043b80775,MISLENE LUSIA OLIVEIRA,12,6338.500000,528.208333
10,26c97a598bf242128d314f605b6e7c38729a09ef54146b53f45eb4fe299cd41d,AURORA LÍDIA DE SANTANA GOMES,12,3622.000000,301.833333


### Working with SQL Alchemy

We'll show a couple examples of how the same tables can be generated using SQL magic within markdown blocks themselves.

We'll not generate all the previously generated tables, but a few selected ones.

The database contains two tables: budgetData and offBudgetData

In [25]:
import prettytable
prettytable.DEFAULT = prettytable.TableStyle.DEFAULT

%load_ext sql
%sql sqlite:///portalDaTransparencia.sqlite


In [26]:
# Cheking the database schema
%sql select * from sqlite_schema

 * sqlite:///portalDaTransparencia.sqlite
Done.


type,name,tbl_name,rootpage,sql
table,budgetData,budgetData,2,"CREATE TABLE ""budgetData"" (""mostrarLicitacao"" INTEGER, ""vlrDescontoInss"" REAL, ""desLicitacaoEmpenhoFormatado"" TEXT, ""desProcessoEmpenho"" TEXT, ""totalDesconto"" REAL, ""numEmpenho"" TEXT, ""vlrLiquidoPago"" REAL, ""desDocumentoFornecedorFormatado"" TEXT, ""vlrDescontoOutros"" REAL, ""vlrPago"" REAL, ""desModalidade"" TEXT, ""desDocumentoFornecedor"" TEXT, ""desAgrupamento"" TEXT, ""desMovimento"" TEXT, ""numAnoEmpenho"" INTEGER, ""vlrLiquidado"" REAL, ""vlrDescontoIrrf"" REAL, ""vlrEmpenhado"" REAL)"
table,offBudgetData,offBudgetData,1999,"CREATE TABLE ""offBudgetData"" (""desClassificacao"" TEXT, ""nomCredor"" TEXT, ""numDocumentoCredor"" TEXT, ""vlrDespesa"" REAL, ""codDespesaExtra"" INTEGER, ""desConta"" TEXT, ""datMovimento"" TEXT, ""numDocumentCredorFormatado"" TEXT)"


Entities that received most budgetary resources from municipality:

In [27]:
%%sql

select
    desDocumentoFornecedor,
    desAgrupamento,
    count(desDocumentoFornecedor) transactions,
    sum(vlrEmpenhado) committedValue,
    (sum(vlrEmpenhado) / count(desDocumentoFornecedor)) averageCommited
from budgetData
group by desDocumentoFornecedor
order by 4 desc
limit 5;

 * sqlite:///portalDaTransparencia.sqlite
Done.


desDocumentoFornecedor,desAgrupamento,transactions,committedValue,averageCommited
02.319.394/0001-70,02.319.394/0001-70 - CISALP CONSORCIO PUBLICO INTERMUNICIPAL DE SAUDE A MICRO REG,7800,172229378.03,22080.68949102564
18.602.078/0001-41,18.602.078/0001-41 - MUNICIPIO DE LAGOA FORMOSA,2897,69441769.18,23970.234442526755
23.096.837/0001-81,23.096.837/0001-81 - SISTEMA DE BENEFICIENCIA DOS SERV PUBL MUN LAGOA FORMOS,1004,45463254.67,45282.12616533865
09.415.124/0001-02,09.415.124/0001-02 - SOLID CONTRUTORA E SERVICOS LTDA,733,26482197.94,36128.510150068214
00.604.122/0001-97,00.604.122/0001-97 - TRIVALE INSTITUICAO DE PAGAMENTO LTDA,494,15469901.5,31315.59008097166


Rank of Entities that had the largest number of transactions within municipality:

In [28]:
%%sql

with aggData as (
    select
        desDocumentoFornecedor,
        desAgrupamento,
        count(desDocumentoFornecedor) transactions,
        sum(vlrEmpenhado) committedValue,
        (sum(vlrEmpenhado) / count(desDocumentoFornecedor)) averageCommited
    from budgetData
    group by desDocumentoFornecedor)
select
    desDocumentoFornecedor,
    transactions,
    rank() over (order by transactions desc) as ranking
from aggData
limit 10;

 * sqlite:///portalDaTransparencia.sqlite
Done.


desDocumentoFornecedor,transactions,ranking
02.319.394/0001-70,7800,1
29.979.036/0001-40,3095,2
18.602.078/0001-41,2897,3
23.096.837/0001-81,1004,4
09.415.124/0001-02,733,5
02.559.007/0001-73,538,6
03.535.357/0001-62,514,7
00.604.122/0001-97,494,8
6ea64432da291cf0ae52cdb77877fd01d5f61a0a72f49f89388cb7caa48b15a5,392,9
20.459.928/0001-46,379,10


Rank of entities that had the largest average value per transaction for the year of 2025:

In [29]:
%%sql

with aggData as (
    select
        desDocumentoFornecedor,
        desAgrupamento,
        count(desDocumentoFornecedor) transactions,
        sum(vlrEmpenhado) committedValue,
        (sum(vlrEmpenhado) / count(desDocumentoFornecedor)) averageCommited
    from budgetData
    where numAnoEmpenho = 2025
    group by desDocumentoFornecedor)
select
    desDocumentoFornecedor,
    averageCommited,
    rank() over (order by averageCommited desc) as ranking
from aggData
limit 10;

 * sqlite:///portalDaTransparencia.sqlite
Done.


desDocumentoFornecedor,averageCommited,ranking
31.534.481/0001-49,1145221.21,1
31.564.854/0001-24,660908.9375,2
47.050.417/0001-22,398067.305,3
42.546.927/0001-71,379000.0,4
49.013.630/0001-90,281227.5,5
21.055.961/0001-73,278828.325,6
30.904.666/0001-35,278640.4428571428,7
41.095.590/0001-60,277594.08,8
02.850.182/0001-15,273141.73000000004,9
41.804.972/0001-16,236271.53818181818,10


We can even a table with the modality preference over time:

In [30]:
%sql select distinct numAnoEmpenho from budgetData;

 * sqlite:///portalDaTransparencia.sqlite
Done.


numAnoEmpenho
2026
2025
2024
2023
2022
2021


In [31]:
%%sql

with agg as (
    select
        desModalidade,
        numAnoEmpenho,
        SUM(vlrEmpenhado) committedValue,
        100.0 * sum(vlrEmpenhado) / SUM(SUM(vlrEmpenhado)) OVER (PARTITION BY numAnoEmpenho) committedPercent
    from budgetData
    group by desModalidade, numAnoEmpenho
)
select
    a.desModalidade,
    round(max(case when numAnoEmpenho = 2021 then a.committedPercent end),2) "2021",
    round(max(case when numAnoEmpenho = 2022 then a.committedPercent end),2) "2022",
    round(max(case when numAnoEmpenho = 2023 then a.committedPercent end),2) "2023",
    round(max(case when numAnoEmpenho = 2024 then a.committedPercent end),2) "2024",
    round(max(case when numAnoEmpenho = 2025 then a.committedPercent end),2) "2025",
    round(max(case when numAnoEmpenho = 2026 then a.committedPercent end),2) "2026",
    round(100.0 * sum(a.committedValue)  / sum(sum(committedValue)) over (),2) allYears
from agg a
group by a.desModalidade
order by allYears desc;

 * sqlite:///portalDaTransparencia.sqlite
Done.


desModalidade,2021,2022,2023,2024,2025,2026,allYears
Dispensa,67.7,54.62,54.42,62.18,66.61,62.24,61.02
Pregao Eletronico,9.09,21.94,18.21,14.01,9.51,6.84,13.6
Pregao,9.44,17.13,13.66,5.06,5.35,5.11,9.16
Concorrencia,None,0.22,2.01,10.41,11.19,17.2,7.2
Tomada de Precos,1.64,3.94,8.19,5.36,2.52,1.83,4.16
Inexigibilidade,4.54,2.05,3.5,2.98,4.82,6.79,4.02
Convite,7.59,0.1,None,None,None,None,0.85


### Working with metabase

Instructions on setting up metabase are out of scope, but you can run it in a docker container using:
```
docker run -d \
  --name metabase \
  --restart unless-stopped \
  -p 3000:3000 \
  -v /home/yourpath:/data:Z \
  -v metabase-data:/metabase-data \
  metabase/metabase:latest
```

Observation: the `-v /home/yourpath:/data:Z \` option give the container access to a local folder. This ease the access to the sqlite database file.

After starting the container, you can access it locally using `https://localhost:3000`

After finishing the setup you can add a new database, as shown bellow:

![](documentation/images/03-metabase-01.png)

Them select the database type (sqlite in our case) and insert the file path within the container:

![](documentation/images/04-metabase-02.png)
![](documentation/images/05-metabase-03.png)

After the database is connected you can open it directly and starting playing with the data and to generate data visualizations

![](documentation/images/06-metabase-04.png)
![](documentation/images/07-metabase-05.png)


Them you can even create dashboards with the data you filtered:
![](documentation/images/08-metabase-06.png)
